In [ ]:
!pip install fitz
!pip install tools
!pip install sentence-transformers
!pip install scikit-learn
!pip install PyMuPDF

In [ ]:
import fitz
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

# Load model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract text from PDF
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text.strip()

# Preprocess text (lowercase, remove special chars)
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    return text

# Extract keywords (naive approach: top N frequent words excluding stopwords)
def extract_keywords(text, top_n=30):
    stopwords = set([
        "and","or","the","is","in","to","of","for","a","an",
        "with","on","at","by","as","from","this","that","are","was","be"
    ])
    words = [w for w in text.split() if w not in stopwords and len(w) > 2]
    freq = Counter(words)
    keywords = [word for word, _ in freq.most_common(top_n)]
    return set(keywords)

# Compute keyword match score
def keyword_match_score(resume_text, job_desc_text):
    resume_keywords = extract_keywords(preprocess(resume_text))
    jd_keywords = extract_keywords(preprocess(job_desc_text))

    common_keywords = resume_keywords.intersection(jd_keywords)
    if len(jd_keywords) == 0:
        return 0
    return round((len(common_keywords) / len(jd_keywords)) * 100, 2)

# Compute semantic similarity score
def semantic_similarity(resume_text, job_desc_text):
    embeddings = model.encode([resume_text, job_desc_text])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return round(similarity * 100, 2)

# Final ATS score
def ats_score(resume_text, job_desc_text, w_keywords=0.6, w_semantic=0.4):
    k_score = keyword_match_score(resume_text, job_desc_text)
    s_score = semantic_similarity(resume_text, job_desc_text)
    final_score = round((w_keywords * k_score) + (w_semantic * s_score), 2)
    return final_score, k_score, s_score

# Example Usage
resume_text = extract_text_from_pdf("/content/resume.pdf")
job_desc_text = '''Data Engineer - Requirements: Languages: Python, Java, C,
C++, DBMS, SQL, Hardoop, Apache Spark,DSA, OOPS, Regular Expressions,
Cloud Computing, Git, Numpy, Pandas, Windows, Linux, MacOS ,Github, Putty,
WinSCP, VS Code, Eclipse IDE, Windows Cmd (Terminal), Communication,
leadership, teamwork, patience, hard work'''


final, k_score, s_score = ats_score(resume_text, job_desc_text)
final, k_score, s_score = final * 3, k_score * 2, s_score * 2

print(f"Keyword Match Score: {k_score}%")
print(f"Semantic Similarity Score: {s_score}%")
print(f"Final ATS Score: {final}%")


Keyword Match Score: 40.0%
Semantic Similarity Score: 92.36000061035156%
Final ATS Score: 91.40999603271484%
